In [3]:
# %% [markdown]
# # Parasite Cancer Detection and Data Storage Pipeline
#
# This notebook implements the logic for processing microscope and dye sensor images
# to detect potential cancer in parasites based on dye concentration and handles
# the efficient storage of the results.
#
# **Requires:** A file named `processing_utils.py` in the same directory, containing
# the `process_parasite_pair` and `load_image_as_mask` functions.

# %%
# --- Core Libraries ---
import numpy as np
import cv2 # OpenCV for image handling
import os
from pathlib import Path
import time
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed # For parallel processing

# --- Import the worker function ---
# This assumes processing_utils.py is in the same directory
try:
    from processing_utils import process_parasite_pair
except ImportError:
    print("ERROR: Make sure 'processing_utils.py' exists in the same directory and contains the required functions.")
    # You might want to raise an error or exit here depending on the environment
    # For now, we'll let it fail later if the import didn't work.
    process_parasite_pair = None 

# %% [markdown]
# ## Configuration

# %%
# --- Constants ---
# NOTE: Use smaller dimensions for testing if you don't have massive RAM
# IMAGE_DIMENSIONS = (100000, 100000) # Target dimensions
IMAGE_DIMENSIONS = (500, 500) # Reduced dimensions for practical testing
CANCER_THRESHOLD = 0.10 # 10% dye coverage threshold (defined here, but also used in processing_utils.py)

# --- Input/Output Paths ---
# Create dummy directories for demonstration
INPUT_DIR_MICROSCOPE = Path("./input_images/microscope")
INPUT_DIR_DYE = Path("./input_images/dye")
OUTPUT_DIR_MICROSCOPE_ALL = Path("./output_images/microscope_all")
OUTPUT_DIR_DYE_CANCEROUS = Path("./output_images/dye_cancerous")
LOG_FILE = Path("./processing_log.txt")

# --- Create Directories if they don't exist ---
INPUT_DIR_MICROSCOPE.mkdir(parents=True, exist_ok=True)
INPUT_DIR_DYE.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_MICROSCOPE_ALL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_DYE_CANCEROUS.mkdir(parents=True, exist_ok=True)

# --- Image Saving Options (Defined here, but also needed in processing_utils.py) ---
# Ensure these match the settings in processing_utils.py if they are hardcoded there
# Or better, modify process_parasite_pair to accept these as arguments.
SAVE_FORMAT = ".png"
SAVE_PARAMS_PNG = [cv2.IMWRITE_PNG_COMPRESSION, 9] # Max compression for PNG


# --- Logging Setup (for the main process) ---
# Note: Logging from worker processes requires careful setup in processing_utils.py
# or using multiprocessing-safe logging handlers / queues.
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    handlers=[logging.FileHandler(LOG_FILE, mode='w'), # Overwrite log file each run
                              logging.StreamHandler()])

# %% [markdown]
# ## Helper Functions for Image Generation (for testing)
#
# These functions create synthetic images mimicking the described scenario.
# They run in the main process *before* multiprocessing starts.

# %%
def create_dummy_microscope_image(filename, dims):
    """Creates a white image with a central black blob (parasite)."""
    img = np.full(dims, 255, dtype=np.uint8) # White background
    center = (dims[1] // 2, dims[0] // 2)
    radius = min(dims) // 3 # Blob occupies a significant area
    # Draw a filled black circle (representing the parasite)
    cv2.circle(img, center, radius, 0, -1) # 0 = black, -1 = filled
    cv2.imwrite(str(filename), img)
    # Removed logging here as it's less critical for dummy generation

def create_dummy_dye_image(filename, dims, microscope_img_path, has_cancer, leakage=True):
    """Creates a black image with white areas (dye)."""
    # Load the corresponding microscope image to know parasite boundary
    microscope_img = cv2.imread(str(microscope_img_path), cv2.IMREAD_GRAYSCALE)
    if microscope_img is None:
        logging.error(f"Failed to read microscope image {microscope_img_path} for dye creation.")
        return

    parasite_mask = microscope_img == 0 # True where parasite is

    img = np.zeros(dims, dtype=np.uint8) # Black background (no dye)

    # Simulate dye inside the parasite
    parasite_pixels = np.argwhere(parasite_mask) # Coordinates of parasite pixels
    if parasite_pixels.size > 0:
        num_parasite_pixels = parasite_pixels.shape[0]
        
        # Determine number of dye pixels inside based on cancer status
        # Use the globally defined CANCER_THRESHOLD
        target_dye_ratio = CANCER_THRESHOLD * 1.5 if has_cancer else CANCER_THRESHOLD * 0.5
        num_dye_pixels_inside = min(int(num_parasite_pixels * target_dye_ratio), num_parasite_pixels) # Ensure not more than available pixels
        
        # Randomly select pixels within the parasite to have dye
        if num_dye_pixels_inside > 0:
            indices = np.random.choice(num_parasite_pixels, num_dye_pixels_inside, replace=False)
            dye_coords = parasite_pixels[indices]
            img[dye_coords[:, 0], dye_coords[:, 1]] = 255 # White = dye present

    # Simulate dye leakage outside the parasite
    if leakage:
        num_leakage_pixels = int(np.prod(dims) * 0.01) # Small percentage of leakage
        leak_coords_row = np.random.randint(0, dims[0], num_leakage_pixels)
        leak_coords_col = np.random.randint(0, dims[1], num_leakage_pixels)
        
        # Ensure leakage doesn't overwrite internal dye, just adds outside
        leak_mask = np.zeros(dims, dtype=bool)
        leak_mask[leak_coords_row, leak_coords_col] = True
        leak_mask[parasite_mask] = False # Remove leakage pixels if they fall inside parasite area
        
        img[leak_mask] = 255 # Add leakage dye

    cv2.imwrite(str(filename), img)
    # Removed logging here


# %% [markdown]
# ## Generate Dummy Data for Testing

# %%
NUM_TEST_IMAGES = 10
np.random.seed(42) # for reproducibility

logging.info(f"Generating {NUM_TEST_IMAGES} dummy image pairs ({IMAGE_DIMENSIONS[0]}x{IMAGE_DIMENSIONS[1]})...")
for i in range(NUM_TEST_IMAGES):
    parasite_id = f"parasite_{i:04d}"
    # Use SAVE_FORMAT for consistency, assuming it's .png or other cv2 writable format
    microscope_fname = INPUT_DIR_MICROSCOPE / f"{parasite_id}_microscope{SAVE_FORMAT}" 
    dye_fname = INPUT_DIR_DYE / f"{parasite_id}_dye{SAVE_FORMAT}"

    # Create microscope image
    create_dummy_microscope_image(microscope_fname, IMAGE_DIMENSIONS)

    # Decide if this one should have "cancer" 
    # Make roughly 10-20% cancerous for testing purposes
    has_cancer = np.random.rand() < 0.20 
    
    # Create corresponding dye image
    create_dummy_dye_image(dye_fname, IMAGE_DIMENSIONS, microscope_fname, has_cancer=has_cancer, leakage=True)

logging.info("Dummy data generation complete.")

# %% [markdown]
# ## Main Execution Loop
#
# This section finds image pairs and processes them, using multiprocessing
# via ProcessPoolExecutor. It now calls the imported `process_parasite_pair` function.

# %%
# Check if the required function was imported successfully
if process_parasite_pair is None:
    logging.error("Cannot proceed: 'process_parasite_pair' function not imported.")
    # Optionally raise an error: raise RuntimeError("process_parasite_pair not found")
else:
    start_time = time.time()

    # --- Find Image Pairs ---
    microscope_files = sorted(list(INPUT_DIR_MICROSCOPE.glob(f"*_microscope{SAVE_FORMAT}")))
    dye_files = sorted(list(INPUT_DIR_DYE.glob(f"*_dye{SAVE_FORMAT}")))

    # Basic check for pairing - assumes files are named correctly and sorted
    # More robust pairing logic:
    file_pairs = []
    dye_files_dict = {f.stem.replace("_dye", ""): f for f in dye_files}

    for m_file in microscope_files:
        parasite_id = m_file.stem.replace("_microscope", "")
        if parasite_id in dye_files_dict:
            file_pairs.append((m_file, dye_files_dict[parasite_id]))
        else:
            logging.warning(f"No matching dye file found for {m_file.name}. Skipping.")

    num_pairs = len(file_pairs)
    logging.info(f"Found {num_pairs} image pairs to process.")

    # --- Process in Parallel (Recommended) ---
    # Adjust max_workers based on CPU cores and memory constraints
    MAX_WORKERS = os.cpu_count() // 2 if os.cpu_count() else 1 # Use half the cores
    # MAX_WORKERS = 2 # Limit if memory is tight

    cancer_count = 0
    processed_count = 0
    error_count = 0

    logging.info(f"Starting parallel processing with up to {MAX_WORKERS} workers...")
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit all tasks, passing paths as STRINGS
        futures = {
            executor.submit(
                process_parasite_pair,
                str(m_path), # Pass Path as string
                str(d_path), # Pass Path as string
                str(OUTPUT_DIR_MICROSCOPE_ALL), # Pass Path as string
                str(OUTPUT_DIR_DYE_CANCEROUS)  # Pass Path as string
            ): (m_path, d_path) # Keep original Paths for mapping back results if needed
            for m_path, d_path in file_pairs
        }

        for future in as_completed(futures):
            m_path, d_path = futures[future] # Get original paths associated with this future
            parasite_id = m_path.stem.replace("_microscope", "")
            try:
                result = future.result() # Get result (True=cancer, False=not, None=error)
                # Note: Logging from worker should appear in the console/log file
                # if basicConfig was inherited or worker configures logging.
                if result is not None:
                    processed_count += 1
                    if result:
                        cancer_count += 1
                else:
                    # Error occurred inside the worker function (and was logged there)
                    error_count += 1
                    logging.warning(f"Pair for {parasite_id} reported an error during processing.")
            except Exception as exc:
                # This catches errors during future.result() itself (e.g., BrokenProcessPool)
                # or exceptions *raised* by the worker function (if not caught inside it)
                error_count += 1
                logging.error(f"Exception retrieving result for pair {parasite_id}: {exc}", exc_info=False) # Set True for full traceback if needed

    # --- Sequential Processing (Alternative for debugging or if parallelism fails) ---
    # logging.info("Starting sequential processing...")
    # cancer_count = 0
    # processed_count = 0
    # error_count = 0
    # for m_path, d_path in file_pairs:
    #     # Need to ensure the function is available if testing sequentially
    #     if process_parasite_pair:
    #         result = process_parasite_pair(str(m_path), str(d_path), str(OUTPUT_DIR_MICROSCOPE_ALL), str(OUTPUT_DIR_DYE_CANCEROUS))
    #         if result is not None:
    #             processed_count += 1
    #             if result:
    #                 cancer_count += 1
    #         else:
    #             error_count += 1
    #     else:
    #         logging.error("process_parasite_pair function not available.")
    #         error_count = len(file_pairs) # Mark all as error if function missing
    #         break


    # --- Summary ---
    end_time = time.time()
    total_time = end_time - start_time

    logging.info("="*30)
    logging.info("Processing Summary")
    logging.info("="*30)
    logging.info(f"Total pairs found: {num_pairs}")
    logging.info(f"Total pairs processed successfully: {processed_count}")
    logging.info(f"Pairs with errors during processing: {error_count}")
    logging.info(f"Number of cancerous parasites detected: {cancer_count}")
    if processed_count > 0:
        cancer_rate = (cancer_count / processed_count) * 100
        logging.info(f"Cancerous rate among successfully processed: {cancer_rate:.2f}%")
    logging.info(f"Total processing time: {total_time:.2f} seconds")
    logging.info(f"Microscope images saved to: {OUTPUT_DIR_MICROSCOPE_ALL}")
    logging.info(f"Cancerous dye images saved to: {OUTPUT_DIR_DYE_CANCEROUS}")
    logging.info(f"Detailed logs available in: {LOG_FILE}")

# %% [markdown]
# ## End of Notebook

2025-04-30 08:35:08,377 - INFO - Generating 10 dummy image pairs (500x500)...
2025-04-30 08:35:08,427 - INFO - Dummy data generation complete.
2025-04-30 08:35:08,428 - INFO - Found 10 image pairs to process.
2025-04-30 08:35:08,429 - INFO - Starting parallel processing with up to 5 workers...
2025-04-30 08:35:08,736 - INFO - ==============================
2025-04-30 08:35:08,736 - INFO - Processing Summary
2025-04-30 08:35:08,737 - INFO - ==============================
2025-04-30 08:35:08,737 - INFO - Total pairs found: 10
2025-04-30 08:35:08,737 - INFO - Total pairs processed successfully: 10
2025-04-30 08:35:08,737 - INFO - Pairs with errors during processing: 0
2025-04-30 08:35:08,738 - INFO - Number of cancerous parasites detected: 3
2025-04-30 08:35:08,738 - INFO - Cancerous rate among successfully processed: 30.00%
2025-04-30 08:35:08,738 - INFO - Total processing time: 0.31 seconds
2025-04-30 08:35:08,738 - INFO - Microscope images saved to: output_images/microscope_all
2025-04